In [1]:
# Import libraries here

import numpy as np
import pandas as pd
import os
import sys
import scipy.optimize as sco

In [2]:
current_dir = os.path.abspath('')
project_root = os.path.abspath(os.path.join(current_dir, '../../'))

if project_root not in sys.path:
    sys.path.append(project_root)

os.chdir(project_root)

print(f"Working directory set to: {os.getcwd()}")

Working directory set to: D:\users\kamen.dimitrov\desktop\softuni\math_concepts_for_developers\08_final_project


In [3]:
# Import project modules here

import importlib
from src.portfolio_simulation_utils import portfolio_simulation as p_sim
from src.data_pipeline_utils import data_fetching_handling as data_pipe
importlib.reload(p_sim)
importlib.reload(data_pipe)

<module 'src.data_pipeline_utils.data_fetching_handling' from 'D:\\users\\kamen.dimitrov\\desktop\\softuni\\math_concepts_for_developers\\08_final_project\\src\\data_pipeline_utils\\data_fetching_handling.py'>

## Harry Markowitz Efficient Frontier ##

Let us now repeat the exercise with the same stocks, however using more robust mathematical modeling skills. While I am building this project, I am very curious if applying the mathematical concepts more carefully will lead to a result, which is similar enough to the practical approach I have utilized in the past. 

The first step is the use the exact same tickers and rebuild our returns dataframe, very similar to the practical approach

In [4]:
tickers = ['AAPL', 'NVDA', 'MSFT', 'JNJ', 'BAC', 'VZ', 'WMT', 'UPS', 'PFE', 'JPM']

In [5]:
returns_df =  data_pipe.build_returns_df(tickers)

### TODO -> add description ###

1. Compute the Input Matrices (Data Pipeline)Your current pipeline fetches historical prices. You must convert these into a mean returns vector and a covariance matrix. These are the fundamental inputs for the analytical approach.Expected Returns Vector ($\mu$): Calculate the annualized mean of the daily returns for each asset.Covariance Matrix ($\Sigma$): Calculate the annualized covariance of the daily returns between all asset pairs.

In [6]:
trading_days = 252
mean_returns = returns_df.mean() * trading_days
cov_matrix = returns_df.cov() * trading_days

In [7]:
print(mean_returns)

AAPL    0.252885
NVDA    0.554150
MSFT    0.216151
JNJ     0.111563
BAC     0.164240
VZ      0.049584
WMT     0.190467
UPS     0.053391
PFE     0.036996
JPM     0.193015
dtype: float64


In [8]:
print(cov_matrix)

          AAPL      NVDA      MSFT       JNJ       BAC        VZ       WMT  \
AAPL  0.083425  0.077180  0.051326  0.015951  0.037031  0.011787  0.019635   
NVDA  0.077180  0.244134  0.081743  0.009668  0.049877  0.005035  0.021675   
MSFT  0.051326  0.081743  0.072250  0.014847  0.033983  0.010490  0.018263   
JNJ   0.015951  0.009668  0.014847  0.033561  0.017613  0.014191  0.012812   
BAC   0.037031  0.049877  0.033983  0.017613  0.094842  0.017877  0.015946   
VZ    0.011787  0.005035  0.010490  0.014191  0.017877  0.040688  0.011994   
WMT   0.019635  0.021675  0.018263  0.012812  0.015946  0.011994  0.046595   
UPS   0.033188  0.042960  0.028776  0.015043  0.036413  0.014351  0.016449   
PFE   0.019717  0.016864  0.018367  0.022114  0.023413  0.015807  0.012335   
JPM   0.032782  0.045580  0.031153  0.016845  0.074374  0.015640  0.014122   

           UPS       PFE       JPM  
AAPL  0.033188  0.019717  0.032782  
NVDA  0.042960  0.016864  0.045580  
MSFT  0.028776  0.018367  0.03

### 2. Define the Linear Algebra  ##

Modern Portfolio Theory evaluates portfolios using matrix multiplication. You must define functions to compute portfolio performance given a weight vector ($w$).

Portfolio Return: $E(R_p) = w^T \mu$

Portfolio Variance: $\sigma_p^2 = w^T \Sigma w$

Let us define the starting weights vector by stating an equal weight of each stock in the portfolio for a starting point

In [9]:
number_of_stocks = len(tickers)
weights = np.array([1 / number_of_stocks] * number_of_stocks)

In [10]:
def calc_portfolio_performance(weights, mean_returns, cov_matrix):
    returns = np.dot(weights, mean_returns)
    std_dev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return returns, std_dev

returns, std_dev = calc_portfolio_performance(weights, mean_returns, cov_matrix)

In [11]:
print(returns)
print(std_dev)

0.18224402903701814
0.17764468017644894


In [12]:
def negative_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate=0.03):
    p_ret, p_std = calc_portfolio_performance(weights, mean_returns, cov_matrix)
    return -(p_ret - risk_free_rate) / p_std

In [13]:
num_assets = len(mean_returns)
args = (mean_returns, cov_matrix)

# Constraint: sum of weights equals 1
constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

# Bounds: weights between 0 and 1
bounds = tuple((0, 1) for asset in range(num_assets))

# Initial guess (equal distribution)
initial_guess = num_assets * [1. / num_assets,]

In [14]:
optimal_portfolio = sco.minimize(
    negative_sharpe_ratio, 
    initial_guess, 
    args=args,
    method='SLSQP', 
    bounds=bounds, 
    constraints=constraints
)

optimal_weights = optimal_portfolio.x

In [15]:
print(list(zip(tickers, optimal_weights)))

[('AAPL', np.float64(0.03609453897308872)), ('NVDA', np.float64(0.3321363665793156)), ('MSFT', np.float64(3.403039568888877e-17)), ('JNJ', np.float64(0.14479588828972795)), ('BAC', np.float64(5.759824041329242e-17)), ('VZ', np.float64(0.0)), ('WMT', np.float64(0.40873435306596223)), ('UPS', np.float64(0.0)), ('PFE', np.float64(0.0)), ('JPM', np.float64(0.07823885309190573))]


### Sanity Check ###

We can take the optimal portfolio using the algorithmic solution from above in a list and the optimal portfolio produced by the portfolio simulation done with Monte Carlo methods. As said, the great the number of sim runs is, the closer the result is. Unfortunately my machine takes nearly 20 minutes to do 30 000 runs and I haven't experimented with a greater number of runs. 

In [16]:
list_1 = [('AAPL', np.float64(0.034049343470848877)), ('NVDA', np.float64(0.3104343381125657)), ('MSFT', np.float64(0.01906932084900172)), ('JNJ', np.float64(0.11491035250708069)), ('BAC', np.float64(0.132925265254072)), ('VZ', np.float64(0.014808443416807251)), ('WMT', np.float64(0.28895955150094504)), ('UPS', np.float64(0.0041376952076077915)), ('PFE', np.float64(0.01296804394922916)), ('JPM', np.float64(0.06773764573184182))]
list_2 = [('AAPL', np.float64(0.03626328058785536)), ('NVDA', np.float64(0.33075999733047673)), ('MSFT', np.float64(4.380176776841438e-17)), ('JNJ', np.float64(0.1461062484684475)), ('BAC', np.float64(1.6588293239028218e-17)), ('VZ', np.float64(5.2909066017292616e-17)), ('WMT', np.float64(0.41006090137180956)), ('UPS', np.float64(0.0)), ('PFE', np.float64(0.0)), ('JPM', np.float64(0.07680957224141094))]

for i in range(len(tickers)):
    print(f"Ticker {list_1[i][0]} -> Monte Carlo {list_1[i][1] * 100:.2f} - Math algorithm {list_2[i][1] * 100:.2f}"
    f"\nDifference {(list_1[i][1] * 100 - list_2[i][1] * 100):.2f}")

Ticker AAPL -> Monte Carlo 3.40 - Math algorithm 3.63
Difference -0.22
Ticker NVDA -> Monte Carlo 31.04 - Math algorithm 33.08
Difference -2.03
Ticker MSFT -> Monte Carlo 1.91 - Math algorithm 0.00
Difference 1.91
Ticker JNJ -> Monte Carlo 11.49 - Math algorithm 14.61
Difference -3.12
Ticker BAC -> Monte Carlo 13.29 - Math algorithm 0.00
Difference 13.29
Ticker VZ -> Monte Carlo 1.48 - Math algorithm 0.00
Difference 1.48
Ticker WMT -> Monte Carlo 28.90 - Math algorithm 41.01
Difference -12.11
Ticker UPS -> Monte Carlo 0.41 - Math algorithm 0.00
Difference 0.41
Ticker PFE -> Monte Carlo 1.30 - Math algorithm 0.00
Difference 1.30
Ticker JPM -> Monte Carlo 6.77 - Math algorithm 7.68
Difference -0.91
